# Đóng gói thư viện fine-tune (Internet bật)

In [ ]:
import json, subprocess
from pathlib import Path

OUT = Path("/kaggle/working/thu_vien")
OUT.mkdir(parents=True, exist_ok=True)
# --no-deps: máy đích đã có torch, transformers, huggingface_hub; không tải lại torch cho nhẹ. Cài ở đích bằng pip --no-index --find-links.
GOI = ["peft", "accelerate", "sacrebleu", "portalocker", "regex", "tabulate", "colorama", "lxml", "safetensors", "psutil"]

def sh(c):
    print(">>", c[:300], flush=True)
    r = subprocess.run(c, shell=True, text=True, capture_output=True)
    print((r.stdout or "")[-3000:], (r.stderr or "")[-3000:], flush=True)
    return r.returncode

sh("which uv || pip install -q uv")
assert sh("uv python install 3.12") == 0
assert sh("uv venv /tmp/py312 --python 3.12 --seed") == 0
assert sh(f"/tmp/py312/bin/python -m pip download --no-cache-dir --no-deps -d {OUT} --only-binary=:all: " + " ".join(GOI)) == 0
ws = sorted(OUT.glob("*.whl"))
print(len(ws), "wheel:", [p.name for p in ws], flush=True)
for g in GOI:
    assert any(p.name.lower().startswith(g.replace("-", "_")) for p in ws), f"thieu wheel {g}"
(OUT / "MANIFEST.json").write_text(json.dumps({"goi": GOI, "files": [p.name for p in ws]}, indent=1), encoding="utf-8")
print("XONG", flush=True)
